In [55]:
import os

import pandas as pd

root = "../../data/data_splits/entropy_fallback/"

mmlu_df = pd.read_csv(
    "../../data/source/mmlu_pro_stem.tsv",
    sep="\t",
    header=0,
)

for subdir in os.listdir(root):
    subdir_path = os.path.join(root, subdir)
    if os.path.isdir(subdir_path):
        for file in os.listdir(subdir_path):
            if file.endswith(".tsv"):
                corrupt_path = os.path.join(subdir_path, file)

                corrupt_df = pd.read_csv(
                    corrupt_path,
                    sep="\t",
                    header=0,
                )
                columns_to_drop = [col for col in corrupt_df.columns if col in mmlu_df.columns and col != "question_id"]
                corrupt_df = corrupt_df.drop(columns=columns_to_drop)
                fixed_df = pd.merge(
                    mmlu_df,
                    corrupt_df,
                    on="question_id",
                    how="inner",
                )
                assert len(fixed_df) == len(corrupt_df), (
                    f"Row count mismatch: fixed_df has {len(fixed_df)} rows, corrupt_df has {len(corrupt_df)} rows"
                )

                fixed_df.to_csv(corrupt_path, sep="\t", index=False)
